# 🧱 Модуль 2. Урок 4: Чистая архитектура SQL через CTE (WITH)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_02_sql/lesson_04_subqueries_and_cte.ipynb)

---

## 1. Принцип модульности: Common Table Expressions (CTE)

В реальной работе запросы на 200 строк с 5 уровнями вложенных подзапросов невозможно поддерживать.
Конструкция `WITH` превращает спагетти-код в последовательный пайплайн:


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"d:\vibe_app\main_data_analytics\course\datasets\ecommerce.db")

query = """
-- Шаг 1: Агрегация на уровне клиента
WITH customer_stats AS (
    SELECT 
        customer_id,
        COUNT(DISTINCT order_id) AS orders_count,
        SUM(total_amount) AS total_spent
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'Completed'
    GROUP BY customer_id
),
-- Шаг 2: Расчет порогов для сегментации
benchmark AS (
    SELECT 
        AVG(total_spent) AS avg_ltv,
        AVG(orders_count) AS avg_orders
    FROM customer_stats
)
-- Шаг 3: Финальный отчет с присвоением бизнес-сегментов
SELECT 
    cs.customer_id,
    cs.orders_count,
    ROUND(cs.total_spent, 2) AS total_spent,
    CASE 
        WHEN cs.total_spent > b.avg_ltv * 2 THEN 'VIP Client'
        WHEN cs.orders_count > b.avg_orders THEN 'Loyal Client'
        ELSE 'Regular Client'
    END AS segment
FROM customer_stats cs
CROSS JOIN benchmark b
ORDER BY cs.total_spent DESC
LIMIT 8;
"""

print("КЛИЕНТСКАЯ СЕГМЕНТАЦИЯ ЧЕРЕЗ CTE:")
print(pd.read_sql(query, conn))
